### Thesis main file
- here I will do my main experiments that are relevant for my thesis
- first I will create the model using hxtorch and test shd inference
- continue with weight transfer using alignment
- in the first trial the same hxtorch model is used for model 25/26, keep it simple and most parameters at default

In [1]:
%matplotlib inline
from _static.tutorial.hxsnn_intro_plots import plot_training, plot_compare_traces, plot_mock, plot_scaled_trace, plot_targets
from _static.common.helpers import setup_hardware_client, save_nightly_calibration
setup_hardware_client()

INFO  12:26:14,053  demo_helpers Connection to hxcube7fpga3chip61_1 established


In [2]:
from functools import partial
import matplotlib.pyplot as plt
import numpy as np

import torch
import hxtorch
import hxtorch.snn as hxsnn
from hxtorch.snn.transforms.weight_transforms import linear_saturating

##### sources
- https://github.com/electronicvisions/hxtorch/blob/master/src/pyhxtorch/hxtorch/examples/spiking/yinyang_model.py
- https://github.com/electronicvisions/hxtorch/blob/master/src/pyhxtorch/hxtorch/examples/spiking/mock_scaling.py
- https://github.com/electronicvisions/hxtorch/blob/master/src/pyhxtorch/hxtorch/spiking/transforms/weight_transforms.py
- keep in mind: dependencies

#### hxtorch model basic
- SHD data imported (using roels code for processing)
- import model weights (25/26) as .npz
- limitation: use pre-existing calibration, just forward pass (mock mode)
- alignment missing (weight_scale and time step size for hardware, hxtorch mock -> hxtorch BSS2)
- scale factor for the weights (Roel neuron -> hxtorch), missing, 10.2 used for now
- tau_mem has to match the one from Roels code (significance seen in single neuron testing)
- the entire test set may fit but pytorch is storing additional info for training, gradients (which in this case is not even used)
- torch.no_grad()
- on 64 test samples accuracy was 75%, on the full test set it dropped to 67.4% 
- reminder ! mock mode is software, so most likely layered inference (pls check code), not to be compared with the results for asynchronous inference (which would be hardware)

In [3]:
try:
    from hxtorch.spiking.utils.dynamic_range.weight_scaling import get_weight_scaling
    print("available")
except ImportError as e:
    print("not available:", e)

available


In [4]:
# load data

shd_test = np.load("thesis_import/shd_test_128.npz")
w_25 = np.load("thesis_import/model25_weights.npz")
w_26 = np.load("thesis_import/model26_weights.npz")

In [5]:
print(shd_test.files)
print(shd_test["y"].shape)

['X', 'y']
(2264,)


In [6]:
print(w_25.files)
print(w_25["w0"].shape)

['w0', 'w1', 'w2', 'w3']
(64, 128)


In [7]:
X = shd_test["X"]                    # (2264, 100, 128): samples, time steps, channels
y = shd_test["y"]                    # (2264,): correct class per sample

n = 80                                # start with the first 64 samples, I did= accuracy at 75%
inputs = torch.tensor(X[:n], dtype=torch.float32).permute(1, 0, 2)   # -> (100, 64, 128)
# permute, since hxtorch expects (time steps, batch size, population)
labels = torch.tensor(y[:n])
print(inputs.shape, labels.shape)

torch.Size([100, 80, 128]) torch.Size([80])


In [8]:
print(w_26.files)
print(w_26["w0"].shape)

['w0', 'w1', 'w2', 'w3']
(64, 128)


##### example for model 25, mock mode 

In [9]:
# linear_saturating from Github not defined, tutorial outdated
# I do not understand how the accuracy could be so much differnt since the behaviour of a neuron is unlikely to be subject to noise (to this extend)
# unless: seed variations, differences in how the forward pass, even in software, is computed

In [14]:
tau_mem = 10e-6   # Roel paper
tau_syn = 1e-6    # there is no equivalent in Roels code, but 1e-6 comes close, cf. single neuron testing

# dt unit value is same for roel software and hxtorch hardware

weight_scale = 1.0     # to be measured with the alignment, but later !
model_threshold, bss2_threshold = 0.3, 125   # roels threshold / notebook value
model_leak, bss2_leak = 0., 80
model_reset, bss2_reset = 0., 80

# for hardware the exact number is not so relevant as the alignment later reveals the
# ratio

# scale factor of 10.25 for now to approximate the neuron to neuron differences
# between roels (snntorch) neuron and hxtorch neuron (while its a extension, 
    # the implementation of neurons differ)

# exp = hxsnn.Experiment(dt=1e-6, mock=True)
# # to avoid potentially time consuming just-in-time calibration
# save_nightly_calibration('spiking2_calix-native.pkl')
# exp.default_execution_instance.load_calib("spiking2_calix-native.pkl") 

scale = 10

def one_lif(size, exp):
    return hxsnn.LIF(size=size, experiment=exp, tau_mem=tau_mem, tau_syn=tau_syn,
                     leak=hxsnn.MixedHXModelParameter(model_leak, bss2_leak),
                     reset=hxsnn.MixedHXModelParameter(model_reset, bss2_reset),
                     threshold=hxsnn.MixedHXModelParameter(model_threshold, bss2_threshold))

# partial() from tutorial/github, linear_saturating and weight_scale, cf. Simon
def one_syn(n_in, n_out, exp): 
    return hxsnn.Synapse(in_features=n_in, out_features=n_out, experiment=exp,
                         transform=partial(linear_saturating, scale=weight_scale))

def run_batch(x_batch, model):
    """Runs one batch of samples through the network and returns the predicted classes."""
    exp = hxsnn.Experiment(dt=1e-6, mock=True)

    syn1, lif1 = one_syn(128, 64, exp), one_lif(64, exp)
    syn2, lif2 = one_syn(64, 64, exp),  one_lif(64, exp)
    syn3, lif3 = one_syn(64, 64, exp),  one_lif(64, exp)
    syn4, lif4 = one_syn(64, 20, exp),  one_lif(20, exp)

    syn1.weight.data = torch.tensor(model["w0"] * scale)
    syn2.weight.data = torch.tensor(model["w1"] * scale)
    syn3.weight.data = torch.tensor(model["w2"] * scale)
    syn4.weight.data = torch.tensor(model["w3"] * scale)

    inputs = torch.tensor(x_batch, dtype=torch.float32).permute(1, 0, 2)   # (time, batch, 128)

    with torch.no_grad():                               # inference only: no gradient record (which occupied most of memory)
        z1 = lif1(syn1(hxsnn.LIFObservables(spikes=inputs)))
        z2 = lif2(syn2(z1))
        z3 = lif3(syn3(z2))
        z4 = lif4(syn4(z3))
        hxsnn.run(exp, inputs.shape[0])

    counts = z4.spikes.sum(dim=0)                       # spikes per sample and output neuron
    return counts.argmax(dim=1) 


batch_size = 128
all_predictions = []

for start in range(0, len(y), batch_size):              # 0, 128, 256, ...
    x_batch = X[start:start + batch_size]
    all_predictions.append(run_batch(x_batch, w_25))
    print("done:", min(start + batch_size, len(y)), "of", len(y))

pred = torch.cat(all_predictions)                       
accuracy = (pred == torch.tensor(y)).float().mean().item()
print("accuracy on the full test set:", accuracy)

done: 128 of 2264
done: 256 of 2264
done: 384 of 2264
done: 512 of 2264
done: 640 of 2264
done: 768 of 2264
done: 896 of 2264
done: 1024 of 2264
done: 1152 of 2264
done: 1280 of 2264
done: 1408 of 2264
done: 1536 of 2264
done: 1664 of 2264
done: 1792 of 2264
done: 1920 of 2264
done: 2048 of 2264
done: 2176 of 2264
done: 2264 of 2264
accuracy on the full test set: 0.6740282773971558


In [15]:
# for model 26 (asynchronous training)
batch_size = 128
all_predictions = []

for start in range(0, len(y), batch_size):              # 0, 128, 256, ...
    x_batch = X[start:start + batch_size]
    all_predictions.append(run_batch(x_batch, w_26))
    print("done:", min(start + batch_size, len(y)), "of", len(y))

pred = torch.cat(all_predictions)                       
accuracy = (pred == torch.tensor(y)).float().mean().item()
print("accuracy on the full test set:", accuracy)

done: 128 of 2264
done: 256 of 2264
done: 384 of 2264
done: 512 of 2264
done: 640 of 2264
done: 768 of 2264
done: 896 of 2264
done: 1024 of 2264
done: 1152 of 2264
done: 1280 of 2264
done: 1408 of 2264
done: 1536 of 2264
done: 1664 of 2264
done: 1792 of 2264
done: 1920 of 2264
done: 2048 of 2264
done: 2176 of 2264
done: 2264 of 2264
accuracy on the full test set: 0.6943463087081909


##### in Roels software environment: 
- F = 128, model 25: 76.3% and model 26: 80.2%
- hxtorch processes all spikes synchronously, without a scheduler, which is closest to F = 128
- still, great loss, but technically is the "true" comparison only possible with F=64
- hxtorch adds one step of delay every layer (cf. single neuron testing), snntorch (basic) neuron, used by Roel has no refractory time
##### refractory_time=0. was the issue :/

In [9]:
tau_mem = 10e-6   # Roel paper
tau_syn = 1e-6    # there is no equivalent in Roels code, but 1e-6 comes close, cf. single neuron testing

# dt unit value is same for roel software and hxtorch hardware

weight_scale = 1.0     # to be measured with the alignment, but later !
model_threshold, bss2_threshold = 0.3, 125   # roels threshold / notebook value
model_leak, bss2_leak = 0., 80
model_reset, bss2_reset = 0., 80

# for hardware the exact number is not so relevant as the alignment later reveals the
# ratio

# scale factor of 10.25 for now to approximate the neuron to neuron differences
# between roels (snntorch) neuron and hxtorch neuron (while its a extension, 
    # the implementation of neurons differ)

# exp = hxsnn.Experiment(dt=1e-6, mock=True)
# # to avoid potentially time consuming just-in-time calibration
# save_nightly_calibration('spiking2_calix-native.pkl')
# exp.default_execution_instance.load_calib("spiking2_calix-native.pkl") 

scale = 10

def one_lif(size, exp):
    return hxsnn.LIF(size=size, experiment=exp, tau_mem=tau_mem, tau_syn=tau_syn,
                     leak=hxsnn.MixedHXModelParameter(model_leak, bss2_leak),
                     reset=hxsnn.MixedHXModelParameter(model_reset, bss2_reset),
                     refractory_time=0.,
                     threshold=hxsnn.MixedHXModelParameter(model_threshold, bss2_threshold))

# partial() from tutorial/github, linear_saturating and weight_scale, cf. Simon
def one_syn(n_in, n_out, exp): 
    return hxsnn.Synapse(in_features=n_in, out_features=n_out, experiment=exp,
                         transform=partial(linear_saturating, scale=weight_scale))

def run_batch(x_batch, model):
    """Runs one batch of samples through the network and returns the predicted classes."""
    exp = hxsnn.Experiment(dt=1e-6, mock=True)

    syn1, lif1 = one_syn(128, 64, exp), one_lif(64, exp)
    syn2, lif2 = one_syn(64, 64, exp),  one_lif(64, exp)
    syn3, lif3 = one_syn(64, 64, exp),  one_lif(64, exp)
    syn4, lif4 = one_syn(64, 20, exp),  one_lif(20, exp)

    syn1.weight.data = torch.tensor(model["w0"] * scale)
    syn2.weight.data = torch.tensor(model["w1"] * scale)
    syn3.weight.data = torch.tensor(model["w2"] * scale)
    syn4.weight.data = torch.tensor(model["w3"] * scale)

    inputs = torch.tensor(x_batch, dtype=torch.float32).permute(1, 0, 2)   # (time, batch, 128)

    with torch.no_grad():                               # inference only: no gradient record (which occupied most of memory)
        z1 = lif1(syn1(hxsnn.LIFObservables(spikes=inputs)))
        z2 = lif2(syn2(z1))
        z3 = lif3(syn3(z2))
        z4 = lif4(syn4(z3))
        hxsnn.run(exp, inputs.shape[0])

    counts = z4.spikes.sum(dim=0)                       # spikes per sample and output neuron
    return counts.argmax(dim=1) 


batch_size = 128
all_predictions = []

for start in range(0, len(y), batch_size):              # 0, 128, 256, ...
    x_batch = X[start:start + batch_size]
    all_predictions.append(run_batch(x_batch, w_25))
    print("done:", min(start + batch_size, len(y)), "of", len(y))

pred = torch.cat(all_predictions)                       
accuracy = (pred == torch.tensor(y)).float().mean().item()
print("accuracy on the full test set:", accuracy) # out of memory again, unless I restart kernel for each cell... suggestions?

done: 128 of 2264
done: 256 of 2264
done: 384 of 2264
done: 512 of 2264
done: 640 of 2264
done: 768 of 2264
done: 896 of 2264
done: 1024 of 2264
done: 1152 of 2264
done: 1280 of 2264
done: 1408 of 2264
done: 1536 of 2264
done: 1664 of 2264
done: 1792 of 2264
done: 1920 of 2264
done: 2048 of 2264
done: 2176 of 2264
done: 2264 of 2264
accuracy on the full test set: 0.7619258165359497


In [10]:
# for model 26 (asynchronous training)
batch_size = 128
all_predictions = []

for start in range(0, len(y), batch_size):              # 0, 128, 256, ...
    x_batch = X[start:start + batch_size]
    all_predictions.append(run_batch(x_batch, w_26))
    print("done:", min(start + batch_size, len(y)), "of", len(y))

pred = torch.cat(all_predictions)                       
accuracy = (pred == torch.tensor(y)).float().mean().item()
print("accuracy on the full test set:", accuracy)

done: 128 of 2264
done: 256 of 2264
done: 384 of 2264
done: 512 of 2264
done: 640 of 2264
done: 768 of 2264
done: 896 of 2264
done: 1024 of 2264
done: 1152 of 2264
done: 1280 of 2264
done: 1408 of 2264
done: 1536 of 2264
done: 1664 of 2264
done: 1792 of 2264
done: 1920 of 2264
done: 2048 of 2264
done: 2176 of 2264
done: 2264 of 2264
accuracy on the full test set: 0.8003533482551575


##### partial alignment (weight scale factor, tau_sys): relevant for hxtorch on hardware